In [13]:
import numpy as np
import pandas as pd
import time

pd.set_option('display.precision',6)
pd.set_option('display.max_rows',20)

#### **#Task 1: Aligned vs Misaligned Series Operations**

In [7]:
# Create two Series with FULL overlap
aapl_aligned = pd.Series(
    [150.0, 152.3, 151.8, 153.2],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    name='AAPL'
)

msft_aligned = pd.Series(
    [180.0, 182.1, 181.5, 183.0],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    name='MSFT'
)

# Create a Series with PARTIAL overlap
tsla_misaligned = pd.Series(
    [2800.0, 2820.0, 2790.0],
    index=['2024-01-02', '2024-01-04', '2024-01-05'],  # Missing Jan 3rd
    name='TSLA'
)

In [ ]:
# 1.1 Computing the spread with aligned data
print("1.1 Computing the spread with aligned data")
pread_aligned = aapl_aligned - msft_aligned
print(spread_aligned)

#1.2 Computing the misaligned spread
print("\n1.2 Computing the misaligned spread")
spread_misaligned = aapl_aligned - tsla_misaligned
print(spread_misaligned) #we can see that 2024-01-03 presents a Nan - this is because TSLA doesn't have data for that index


#1.3 Computing the mean of both spreads
print("\n1.3 Computing the mean of both spreads")
print("Mean of aligned spreads: ", spread_aligned.mean())
print("Mean of misaligned spreads: ",spread_misaligned.mean())
print("Mean of misaligned spreads but using .nanmean(): ", np.nanmean(spread_misaligned.to_numpy()))
# Skip NaN is true by default in the .mean()
print("Mean of misaligned spreads: ",spread_misaligned.mean(0,False )))

#1.4 Explicity check for NaNs
print("\n1.4 Explicity check for NaNs")
print("Spread Aligned")
print(spread_aligned.isna())
print(spread_aligned.isna().sum())
print("Spread Misaligned")
print(spread_misaligned.isna())
print(spread_misaligned.isna().sum())

#1.5 Creating a completely non-overlapping Series
print("\n1.5 Creating a completely non-overlapping Series")
goog = pd.Series([2900, 2950], index=['2024-01-08', '2024-01-09'])
spread_non_overlapping = aapl_aligned - goog
print(spread_non_overlapping)
print(spread_non_overlapping.isna())
"""
We can see the it combines the index (you can see apple and google's index's combined...
But if we run a .isna() they are all NaN's.
"""

1.1 Computing the spread with aligned data
2024-01-02   -30.0
2024-01-03   -29.8
2024-01-04   -29.7
2024-01-05   -29.8
dtype: float64

1.2 Computing the misaligned spread
2024-01-02   -2650.0
2024-01-03       NaN
2024-01-04   -2668.2
2024-01-05   -2636.8
dtype: float64

1.3 Computing the mean of both spreads
Mean of aligned spreads:  -29.824999999999996
Mean of misaligned spreads:  -2651.6666666666665
Mean of misaligned spreads but using .nanmean():  -2651.6666666666665

1.4 Explicity check for NaNs
Spread Aligned
2024-01-02    False
2024-01-03    False
2024-01-04    False
2024-01-05    False
dtype: bool
0
Spread Misaligned
2024-01-02    False
2024-01-03     True
2024-01-04    False
2024-01-05    False
dtype: bool
1

1.5 Creating a completely non-overlapping Series
2024-01-02   NaN
2024-01-03   NaN
2024-01-04   NaN
2024-01-05   NaN
2024-01-08   NaN
2024-01-09   NaN
dtype: float64
2024-01-02    True
2024-01-03    True
2024-01-04    True
2024-01-05    True
2024-01-08    True
2024-01-09  

#### **#Task 2: NumPy Broadcasting vs Pandas Alignment**

In [ ]:
#same data, two formats
prices_pandas = pd.Series([150.0, 152.3, 151.8], index=[0,1,2])
prices_numpy = prices_pandas.to_numpy()

#scalar adjustment
adjustment = 5.0

In [ ]:
#2.1 Numpy operation
print("2.1 Numpy operations: seeing how operations work on Numpy arrays")
adjusted_np = prices_numpy + adjustment
print(adjusted_np)
    # you can see from the output that the adjustment is 'broadcasted' across all elements

#2.2 Pandas operation
print("\n2.2 Pandas operations: seeing how operations work on pandas arrays")
adjusted_pd = prices_pandas + adjustment
print(adjusted_pd)
    # you can see that the adjustment applies the same, but an index is applied (from 0)
adjustment_series = pd.Series([5.0, 10.0], index=[1,2])

#2.3 Pandas alignment operations
print("\n2.3 Pandas Alignment Operations")
adjusted_alignment = prices_pandas + adjustment_series
print(adjusted_alignment)
    # you can see here that the adjustment_series is added only to where the index match, i.e at 1 and 2
    # Nothing is added onto index 0 and thus appears as NaN

#2.4 Trying to get NumPy broadcasting behaviour in Pandas
print("\n2.4 Getting Numpy Broadcasting behaviour in Pandas")
mixed = prices_pandas + adjustment_series.values()
print(mixed)
    # .values on mixed-type data can give you an object array. .values() is legacy and discouraged anyways...

2.1 Numpy operations: seeing how operations work on Numpy arrays
[155.  157.3 156.8]

2.2 Pandas operations: seeing how operations work on pandas arrays
0    155.0
1    157.3
2    156.8
dtype: float64

2.3 Pandas Alignment Operations
0      NaN
1    157.3
2    161.8
dtype: float64

2.4 Getting Numpy Broadcasting behaviour in Pandas


TypeError: 'numpy.ndarray' object is not callable

#### **#Task 3: Performance Comparision**

In [35]:
# Generating large dataset
n = 1_000_000

# NumPy arrays
np_array1 = np.random.rand(n)
np_array2 = np.random.rand(n)

# Pandas Series (with integer index)
pd_series1 = pd.Series(np_array1)
pd_series2 = pd.Series(np_array2)

#Pandas Series (with DatetimeIndex - more realistic for finance)
dates = pd.date_range('2020-01-01', periods=n, freq='min') # 1 million minutes
pd_series_dt1 = pd.Series(np_array1, index=dates)
pd_series_dt2 = pd.Series(np_array2, index=dates)

In [ ]:
# 3.1 Benchmark element-wise addition with NumPy
%timeit result_np = np_array1 + np_array2

# 3.2 Benchmark element-wise addition with Pandas (integerindex)
%timeit result_pd = pd_series1 + pd_series2

# 3.3 Benchmark element-wise addition with Pandas (DatetimeIndex)
%timeit result_pd_dt = pd_series_dt1 + pd_series_dt2

# 3.4 Benchmark a more complex operations (moving average)
window = 20
    # Numpy version (manual convolution)
%timeit np_ma = np.convolve(np_array1, np.ones(window)/window, mode='valid')

    # Pandas version (built-in rolling)
%timeit pd_ma = pd_series_dt1.rolling(window).mean()

193 μs ± 1.46 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
205 μs ± 2.04 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
206 μs ± 1.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
4.86 ms ± 54.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.16 ms ± 6.01 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### **#Task 4: Building a misaligned return calculator**

##### *Setting up data*

In [45]:
# Simulate realistic stock data with issues
np.random.seed(42)

# AAPL: full data, 252 trading days
dates_aapl = pd.bdate_range('2024-01-01', periods=252)
prices_aapl = 150 * (1 + np.random.randn(252).cumsum() * 0.01)
aapl = pd.Series(prices_aapl, index=dates_aapl, name='AAPL')

# TSLA: started tracking late, only 200 days
dates_tsla = pd.bdate_range('2024-03-15', periods=200)
prices_tsla = 2800 * (1 + np.random.randn(200).cumsum() * 0.015)
tsla = pd.Series(prices_tsla, index=dates_tsla, name='TSLA')

# MSFT: has 10 random missing days (trading halts, data gaps)
dates_msft = pd.bdate_range('2024-01-01', periods=252)
missing_indices = np.random.choice(252, size=10, replace=False)
dates_msft_filtered = dates_msft.delete(missing_indices)
prices_msft = 180 * (1 + np.random.randn(len(dates_msft_filtered)).cumsum() * 0.008)
msft = pd.Series(prices_msft, index=dates_msft_filtered, name='MSFT')

##### *Tasks*

In [ ]:
# 1. Computing simple returns for each stock
print("1. Computing simple returns for each stock")
aapl_ret = aapl.pct_change()
tsla_ret = tsla.pct_change()
msft_ret = msft.pct_change()
#print(aapl_ret)
#print(tsla_ret)
#print(msft_ret)

# 2. Inspecting returns
print("\n2. Inspecting returns")
print(aapl_ret.head()) # first 5 elements
print(aapl_ret.tail()) # last 5 elements
    # the first element is NaN since there is no pct change on the first number!
print(tsla_ret.head())
print(msft_ret.head())

# 3. Combining all three arrays into a DataFrame
print("\n3. Combining all 3 arrays into a DataFrame")
returns_df = pd.DataFrame({
    'AAPL': aapl_ret,
    'TSLA': tsla_ret,
    'MSFT': msft_ret,
})
print("First 10 indexes: \n", returns_df.head(10)) #printing the first 10 elements
    # we can see that in printing the above, TSLA has NaN'set
    # This is because TSLA only started data since 2024-03-15
print("Constricted view: \n", returns_df.loc['2024-03-14':'2024-03-20'])
    # remember bdate is business days so weekends are ignored

# 4. Counting missing values per stock
print("\n4. Counting missing values per stock")
print(returns_df.isna().sum())
print(returns_df.count()) # counts the number of non-NaN values

# 5. Computing correlation matrix
print("\n5. Computing correlation matrix")
print(returns_df.corr())
    # remember pandas skips NaNs
print(returns_df.corr(min_periods=200))
        # there are only 199 non nan values for tesla, so the .corr() shouldn't work for tesla
"""
In pandas, the min_periods parameter in the .corr() method defines the minimum number of non-null (non-NaN) observations required per pair of columns to compute a valid correlation coefficient. 
If a pair of columns has fewer than min_periods matching observations (after dropping rows with NaNs), the resulting correlation value for that pair will be NaN (Not a Number). 
"""

# 6. Writing function to copmute LOG returns

1. Computing simple returns for each stock

2. Inspecting returns
2024-01-01         NaN
2024-01-02   -0.001376
2024-01-03    0.006454
2024-01-04    0.015079
2024-01-05   -0.002284
Freq: B, Name: AAPL, dtype: float64
2024-12-11   -0.006675
2024-12-12    0.018159
2024-12-13    0.004091
2024-12-16   -0.012686
2024-12-17    0.009353
Freq: B, Name: AAPL, dtype: float64
2024-03-15         NaN
2024-03-18    0.015009
2024-03-19   -0.021761
2024-03-20   -0.007090
2024-03-21    0.018681
Freq: B, Name: TSLA, dtype: float64
2024-01-01         NaN
2024-01-02    0.003115
2024-01-03    0.005274
2024-01-04   -0.007241
2024-01-05    0.013175
Name: MSFT, dtype: float64

3. Combining all 3 arrays into a DataFrame
First 10 indexes: 
                 AAPL  TSLA      MSFT
2024-01-01       NaN   NaN       NaN
2024-01-02 -0.001376   NaN  0.003115
2024-01-03  0.006454   NaN  0.005274
2024-01-04  0.015079   NaN -0.007241
2024-01-05 -0.002284   NaN  0.013175
2024-01-08 -0.002289   NaN -0.000307
2024-01-09  0.01

'\nIn pandas, the min_periods parameter in the .corr() method defines the minimum number of non-null (non-NaN) observations required per pair of columns to compute a valid correlation coefficient. \nIf a pair of columns has fewer than min_periods matching observations (after dropping rows with NaNs), the resulting correlation value for that pair will be NaN (Not a Number). \n'